# CS 468 Assignment 4: Emotion Detection with RNN vs LSTM

Build and evaluate **two neural sequence models** for emotion classification:

- **RNN**
- **LSTM**

We will use the Hugging Face dataset **Enhanced_Emotion_Classification_Dataset** and **filter to the `goemotions` source**.

The GoEmotion dataset is a multiple labeled dataset of Reddit posts,
GoEmotions dataset is manually annotated dataset
of 58k English Reddit comments, labeled for
27 emotion categories or Neutral. In the tutorial, we have grouped the 27 emotions into 6 categories, and together with Neutral class, we did a 7-class classification task. In this assignment, you will **remove the `neutral` emotion** and perform **6-class** classification:
`joy, sadness, anger, fear, surprise, disgust`.

**Links**
- Original GoEmotions paper: https://arxiv.org/abs/2005.00547  
- Dataset on Hugging Face: https://huggingface.co/datasets/jiangchengchengNLP/Enhanced_Emotion_Classification_Dataset  

---

## Your tasks (what you must implement / modify)

1. **Remove `neutral`** and train/evaluate only on the remaining **6 emotions**.
2. **Use pretrained GloVe word embeddings** loaded via **Gensim** (e.g., `glove-wiki-gigaword-100`) in your RNN/LSTM models.
3. **Change the model architecture** (examples: bidirectional, more layers, different pooling, attention, layer norm, etc.).
4. **Do more hyperparameter tuning** (expand the search space and justify choices).
5. **Plot training vs validation loss curves** to diagnose overfitting/underfitting.
6. **Report `classification_report()` on the test set** and **compare** the best RNN vs best LSTM.


## 0) Hugging Face access (token setup)
This dataset may require authenticated access or may download more reliably with a Hugging Face token.

### Step A — Create an account
### Step B — Create a token
### Step C — Add token to Google Colab Secrets (recommended)

In [1]:
# (Recommended) Login using your token stored in Colab Secrets as HF_TOKEN
# If you prefer, you can paste manually when prompted.
!pip -q install huggingface_hub==0.24.6
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN", None)
if token is None:                            #
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')

if token is None:
    from getpass import getpass
    token = getpass("Paste your Hugging Face token (input hidden): ")

login(token=token, add_to_git_credential=True)  #login(token=token)
print("✅ Logged in to Hugging Face")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.24.6 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.24.6 which is incompatible.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.24.6 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.24.6 which is incompatible.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful
✅ Logged in to Hugging Face


## 1) Install packages


In [2]:
# Fix NumPy 2.x ABI incompatibilities in Colab by using NumPy 1.26.x
!pip -q uninstall -y numpy scipy gensim pyarrow datasets scikit-learn
!pip -q install --no-cache-dir -U pip setuptools wheel
!pip -q install --no-cache-dir "numpy<1.26.5" "scipy<1.12" "pyarrow<16" gensim==4.3.3 scikit-learn datasets==2.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 194.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 432.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.14 requires numpy>=1.19.0; python_version >= "3.9", which is not installed.
thinc 8.3.13 requires numpy<3.0.0,>=1.21.0, which is not installed.
torchvision 0.25.0+cu128 requires numpy, which is not installed.
fastai 2.8.7 requires scikit-learn, which is not installed.
fastai 2.8.7 requires scipy, which is not installed.
peft 0.19.1 requires numpy>=1.17, which is not installed.
tensorflow 2.20.0 requires numpy>=1.26.0, which is not installed.
cufflinks 0.17.3 requires numpy>=1.9.2, which is not installed.
pandas-gbq 0.30.0 requires numpy>=1.18.1, which is not installed.
pandas-gbq 0.30.0 requires pyarrow>=4.0.0, which is not installed.
osqp 1.1.1 requi

In [3]:
# # As a replacement for above
!pip uninstall -y numpy scipy gensim pyarrow datasets scikit-learn opencv-python opencv-python-headless opencv-contrib-python jax jaxlib

# Only applicable to this assignment
!pip install --no-cache-dir \
  numpy==1.26.4 \
  scipy==1.11.4 \
  gensim==4.3.3 \
  datasets==2.19.1 \
  scikit-learn==1.4.2 \
  matplotlib==3.8.4 \
  torch==2.2.2 \
  huggingface_hub==0.24.6

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.11.4
Uninstalling scipy-1.11.4:
  Successfully uninstalled scipy-1.11.4
Found existing installation: gensim 4.3.3
Uninstalling gensim-4.3.3:
  Successfully uninstalled gensim-4.3.3
Found existing installation: pyarrow 15.0.2
Uninstalling pyarrow-15.0.2:
  Successfully uninstalled pyarrow-15.0.2
Found existing installation: datasets 2.19.1
Uninstalling datasets-2.19.1:
  Successfully uninstalled datasets-2.19.1
Found existing installation: scikit-learn 1.8.0
Uninstalling scikit-learn-1.8.0:
  Successfully uninstalled scikit-learn-1.8.0
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Fo

In [1]:
import numpy as np, scipy, gensim, datasets, pyarrow
print("numpy:", np.__version__, end= "\t")
print("scipy:", scipy.__version__, end= "\t")
print("gensim:", gensim.__version__, end= "\t")
print("datasets:", datasets.__version__, end= "\t")
print("pyarrow:", pyarrow.__version__)
from datasets import load_dataset
print("✅ load_dataset import OK")

numpy: 1.26.4	scipy: 1.11.4	gensim: 4.3.3	datasets: 2.19.1	pyarrow: 24.0.0
✅ load_dataset import OK


In [2]:
# Pin a stable datasets version for classroom reproducibility
# !pip -q install datasets==2.19.1 gensim==4.3.3 scikit-learn==1.4.2 matplotlib==3.8.4 numpy==1.26.4 torch==2.2.2

import re, random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import classification_report, f1_score, accuracy_score
from collections import Counter
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

## 2) Load dataset and filter to GoEmotions
 Because of the datasets package and packages like Numpy version problems, we can't use huggingface's datasets.load_dataset directly to load the data. So we have a workaroud, load the csv files directly.


In [3]:
from datasets import load_dataset
# START CODE
data_files = {
    "train": "hf://datasets/jiangchengchengNLP/Enhanced_Emotion_Classification_Dataset/train/train.csv",
    "validation": "hf://datasets/jiangchengchengNLP/Enhanced_Emotion_Classification_Dataset/val/val.csv",
    "test": "hf://datasets/jiangchengchengNLP/Enhanced_Emotion_Classification_Dataset/test/test.csv",
}
# END CODE

ds = load_dataset("csv", data_files=data_files)
print(ds)
print(ds["train"].column_names)
print("Train/Val/Test:", len(ds["train"]), len(ds["validation"]), len(ds["test"]))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'source'],
        num_rows: 186619
    })
    validation: Dataset({
        features: ['text', 'label_text', 'source'],
        num_rows: 31086
    })
    test: Dataset({
        features: ['text', 'label_text', 'source'],
        num_rows: 22721
    })
})
['text', 'label_text', 'source']
Train/Val/Test: 186619 31086 22721


In [4]:
def is_goemotions(ex):
    return ex["source"] == "goemotions"

ds_go = ds.filter(is_goemotions, num_proc=4)

print("GoEmotions subset sizes:")
for sp in ["train", "validation", "test"]:
    print(f"{sp:>10}: {len(ds_go[sp])}")


Filter (num_proc=4):   0%|          | 0/186619 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/31086 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/22721 [00:00<?, ? examples/s]

GoEmotions subset sizes:
     train: 41251
validation: 11786
      test: 5893


## 3) Remove `neutral` to form a 6-class dataset

We keep only: `joy, sadness, anger, fear, surprise, disgust` (no neutral).


In [5]:
#START CODE HERE
# 2. Remove the neutral class, define TARGET_LABELS_6 only has "joy", "sadness", "anger", "fear", "surprise", "disgust"
TARGET_LABELS_6 = {"joy", "sadness", "anger", "fear", "surprise", "disgust"}
# END CODE HERE

def is_six_class(ex):
    return ex["label_text"] in TARGET_LABELS_6
ds_task = ds_go.filter(is_six_class, num_proc=4)

print("6-class sizes (neutral removed):")
for sp in ["train", "validation", "test"]:
    print(f"{sp:>10}: {len(ds_task[sp])}")


Filter (num_proc=4):   0%|          | 0/41251 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/11786 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/5893 [00:00<?, ? examples/s]

6-class sizes (neutral removed):
     train: 28759
validation: 8357
      test: 4085


## 4) Tokenization + sanity check (remove empty-token rows)

For simplicity, we use a basic tokenizer that keeps only letters/numbers/apostrophes.


In [6]:
TOKEN_RE = re.compile(r"[A-Za-z0-9']+")

def tokenize(text: str):
    if text is None: return []
    text = str(text).strip().lower()
    if not text: return []
    return TOKEN_RE.findall(text)

def has_tokens(ex):
    return len(tokenize(ex["text"])) > 0

def count_empty(split):
    n = 0
    for t in split["text"]:
        if len(tokenize(t)) == 0:
            n += 1
    return n

print("Empty-token examples BEFORE filtering:")
for sp in ["train", "validation", "test"]:
    print(f"{sp:>10}: {count_empty(ds_task[sp])}")

ds_task = ds_task.filter(has_tokens, num_proc=4)

print("\nSizes AFTER filtering empty-token rows:")
for sp in ["train", "validation", "test"]:
    print(f"{sp:>10}: {len(ds_task[sp])}")


Empty-token examples BEFORE filtering:
     train: 0
validation: 0
      test: 0


Filter (num_proc=4):   0%|          | 0/28759 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/8357 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/4085 [00:00<?, ? examples/s]


Sizes AFTER filtering empty-token rows:
     train: 28759
validation: 8357
      test: 4085


## 5) Build label mapping (6 classes)


In [7]:
labels = sorted(list(set(ds_task["train"]["label_text"])))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

print(f"Labels: {labels}\nNum labels: {len(labels)}\n")
def add_label_id(ex):
    ex["label"] = label2id[ex["label_text"]]
    return ex

ds_task = ds_task.map(add_label_id, num_proc=4)


Labels: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
Num labels: 6



Map (num_proc=4):   0%|          | 0/28759 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/8357 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/4085 [00:00<?, ? examples/s]

## 6) Build vocabulary from training set


In [8]:
MAX_VOCAB = 50000
MIN_FREQ = 2

counter = Counter()
for t in ds_task["train"]["text"]:
    counter.update(tokenize(t))

PAD, UNK = "<pad>", "<unk>"
vocab = {PAD: 0, UNK: 1}
for tok, freq in counter.most_common():
    if freq < MIN_FREQ:
        break
    if tok in vocab:
        continue
    vocab[tok] = len(vocab)
    if len(vocab) >= MAX_VOCAB:
        break

pad_id, unk_id = vocab[PAD], vocab[UNK]
print("Vocab size:", len(vocab))


Vocab size: 10487


## 7) Load pretrained GloVe embeddings using Gensim


In [9]:
import gensim.downloader as api

GLOVE_NAME = "glove-wiki-gigaword-100" # Trial size for fine tuning: # GLOVE_NAME = "glove-wiki-gigaword-200"

glove = api.load(GLOVE_NAME)
EMBED_DIM = glove.vector_size
print("Loaded", GLOVE_NAME, "dim =", EMBED_DIM)


[==================================================] 100.0% 128.1/128.1MB downloaded
Loaded glove-wiki-gigaword-100 dim = 100


In [10]:
rng = np.random.default_rng(SEED)
embedding_matrix = np.zeros((len(vocab), EMBED_DIM), dtype=np.float32)
oov = 0
for token, idx in vocab.items():
    if token == PAD: continue
    if token in glove:
        embedding_matrix[idx] = glove[token]
    else:
        oov += 1
        embedding_matrix[idx] = rng.normal(0.0, 0.1, size=(EMBED_DIM,))
print("OOV tokens:", oov, "/", len(vocab))


OOV tokens: 497 / 10487


## 8) Encode + PyTorch datasets


In [11]:
MAX_LEN = 64  # tune sequence length               # MAX_LEN = 512# MAX_LEN = 131,072
# determining the max # of tokens the model processes at once

def encode(text: str):
    toks = tokenize(text)[:MAX_LEN]
    ids = [vocab.get(tok, unk_id) for tok in toks]
    length = len(ids)
    if length == 0:
        ids = [unk_id]
        length = 1
    if length < MAX_LEN:
        ids += [pad_id] * (MAX_LEN - length)
    return ids, length

class EmotionDataset(Dataset):
    def __init__(self, hf_split):
        self.texts = hf_split["text"]
        self.labels = hf_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length = encode(self.texts[idx])
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )

train_data = EmotionDataset(ds_task["train"])
val_data   = EmotionDataset(ds_task["validation"])
test_data  = EmotionDataset(ds_task["test"])


In [12]:

BATCH_SIZE = 256 # 64 # 32 # 16
# smaller batches improve regularization and avoids the local minima
# larger batches increase time efficiency, but may have a chance of converging to suboptimal solutions

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)



## 9) Model definitions (RNN and LSTM)

The code cell below defines the RNN model, your task it to define the LSTM model.


In [13]:
class SequenceClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_layers: int,
        num_classes: int,
        dropout: float,
        rnn_type: str,
        embedding_matrix,
        freeze_embeddings: bool = False,
        bidirectional: bool = False, # True            # Something you can do: try True
        pool: str = "last", # "mean" # "max"           # Something you can try: try "mean" or "max"
    ):
        super().__init__()
        self.pool = pool
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = (not freeze_embeddings)

        self.dropout = nn.Dropout(dropout)

        if rnn_type == "rnn":
          self.rnn = nn.RNN(
              embed_dim,
              hidden_dim,
              num_layers=num_layers,
              batch_first=True,
              dropout=dropout if num_layers > 1 else 0.0,
              bidirectional=bidirectional,
          )

        elif rnn_type == "lstm":


          self.rnn = nn.LSTM(
              embed_dim,
              hidden_dim,
              num_layers=num_layers,
              batch_first=True,
              dropout=dropout if num_layers > 1 else 0.0,
              bidirectional=bidirectional,
          )


        self.fc = nn.Linear(hidden_dim * self.num_directions, num_classes)

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        lengths_cpu = lengths.cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, h = self.rnn(packed)

        h_n, _ = h if isinstance(h, tuple) else (h, None)   # (h, _)


        last = torch.cat([h_n[-2], h_n[-1]], dim=-1) if self.bidirectional else h_n[-1]

        if self.pool != "last":
            out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))
            mask = (input_ids != pad_id).unsqueeze(-1)
            out = out * mask

            if self.pool == "mean":
                denom = mask.sum(dim=1).clamp(min=1)
                last = out.sum(dim=1) / denom
            elif self.pool == "max":
                out = out.masked_fill(~mask, float("-inf"))
                last, _ = out.max(dim=1)
            else:
                raise ValueError("pool must be 'last', 'mean', or 'max'")

        last = self.dropout(last)
        logits = self.fc(last)
        return logits


## 10) Training + evaluation utilities



In [14]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)


    loss_fn = nn.CrossEntropyLoss()   #  define loss function nn.CrossEntropyLoss()


    total_loss = 0.0
    all_preds, all_labels = [], []
    for input_ids, lengths, labels_ in loader:
        input_ids = input_ids.to(device)
        lengths   = lengths.to(device)
        labels_   = labels_.to(device)


        # pass the input_ids and lengths to model, and get the logits,
        #   use the logits and true label in labels_to calculate the loss
        logits = model(input_ids, lengths)
        loss = loss_fn(logits, labels_)


        if is_train:
            optimizer.zero_grad()
            loss.backward()       # perform backprogration
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item() * labels_.size(0)
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(labels_.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, macro_f1, acc

def train_model(model, train_loader, val_loader, lr=1e-3, epochs=10, patience=3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_state, best_val_f1, bad = None, -1.0, 0
    history = {
        "train_loss": [], "val_loss": [], "train_f1": [], "val_f1": [], "train_acc": [], "val_acc": [],
    }

    for epoch in range(1, epochs+1):
        tr_loss, tr_f1, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_f1, va_acc = run_epoch(model, val_loader, optimizer=None)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_f1"].append(tr_f1)
        history["val_f1"].append(va_f1)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)


        # print(
        #     f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.4f} acc {tr_acc:.4f} | "
        #     f"val loss {va_loss:.4f} f1 {va_f1:.4f} acc {va_acc:.4f}"
        # )

        if va_f1 > best_val_f1 + 1e-4:
            best_val_f1 = va_f1
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_f1, history

def plot_loss_curves(history, title):
    plt.figure()
    plt.plot(history["train_loss"], label="train loss")
    plt.plot(history["val_loss"], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.show()


## 11) Hyperparameter tuning


In [ ]:
def tune_and_train(rnn_type="rnn"):

    # Expand this grid substantially, at least three more groups of values
    # And, add bacth size as one hyper-parameter as well
    # Expected 7 groups of hyper-parameter values each includes the batch size as well.
    configs = [
        dict(hidden_dim=128, num_layers=1, dropout=0.3, lr=1e-3, freeze=False, bidi=False, pool="last", batch_size=128),
        dict(hidden_dim=256, num_layers=1, dropout=0.4, lr=8e-4, freeze=False, bidi=True,  pool="last", batch_size=256),
        dict(hidden_dim=256, num_layers=2, dropout=0.4, lr=8e-4, freeze=False, bidi=True,  pool="mean", batch_size=256),
        dict(hidden_dim=384, num_layers=2, dropout=0.5, lr=6e-4, freeze=True,  bidi=True,  pool="mean", batch_size=128),
        dict(hidden_dim=128, num_layers=2, dropout=0.3, lr=1e-3, freeze=False, bidi=False, pool="max", batch_size=64),
        dict(hidden_dim=256, num_layers=3, dropout=0.5, lr=7e-4, freeze=False, bidi=True,  pool="mean", batch_size=64),
        dict(hidden_dim=512, num_layers=2, dropout=0.5, lr=5e-4, freeze=True, bidi=True,  pool="last", batch_size=128),
    ]

    best_model, best_cfg, best_val_f1, best_hist = None, None, -1.0, None

    for i, cfg in enumerate(configs, 1):
        print("\n" + "="*80)
        print(f"[{rnn_type.upper()}] Trial {i}/{len(configs)}: {cfg}")

        model = SequenceClassifier(
            vocab_size=len(vocab),
            embed_dim=EMBED_DIM,
            hidden_dim=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            num_classes=len(labels),
            dropout=cfg["dropout"],
            rnn_type=rnn_type,
            embedding_matrix=embedding_matrix,
            freeze_embeddings=cfg["freeze"],
            bidirectional=cfg["bidi"],
            pool=cfg["pool"],
        )

        model, val_f1, hist = train_model(model, train_loader, val_loader, lr=cfg["lr"], epochs=12, patience=3)
        print(f"[{rnn_type.upper()}] Trial {i} best val macro-F1: {val_f1:.4f}")

        plot_loss_curves(hist, title=f"{rnn_type.upper()} Trial {i} loss curves")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model, best_cfg, best_hist = model, cfg, hist

    return best_model, best_cfg, best_val_f1, best_hist


In [ ]:
best_rnn, best_rnn_cfg, best_rnn_val_f1, best_rnn_hist = tune_and_train("rnn")
print("\nBest RNN config:", best_rnn_cfg)
print("Best RNN val macro-F1:", best_rnn_val_f1)


In [ ]:
best_lstm, best_lstm_cfg, best_lstm_val_f1, best_lstm_hist = tune_and_train("lstm")

print("\nBest LSTM config:", best_lstm_cfg)
print("Best LSTM val macro-F1:", best_lstm_val_f1)


## 12) Test evaluation + classification report


In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for input_ids, lengths, labels_ in loader:
        input_ids = input_ids.to(device)
        lengths   = lengths.to(device)
        logits = model(input_ids, lengths)
        preds = torch.argmax(logits, dim=-1).detach().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels_.tolist())
    return all_labels, all_preds

def report(model, name):
    y_true, y_pred = predict(model, test_loader)
    print(f"\n{name} test accuracy:", accuracy_score(y_true, y_pred))
    print(f"{name} test macro-F1:", f1_score(y_true, y_pred, average='macro'))
    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred,
        target_names=[id2label[i] for i in range(len(labels))],
        digits=4
    ))

report(best_rnn, "RNN")
report(best_lstm, "LSTM")


This work shows how sequence models operate on emotion classifiers. The model, LSTM has better performance efficiency because of its ability to capture long-term dependencies and its workaround with the vanishing gradient problem. I thought that hyperparameter tuning excelled by improving the models performance; which the model adjusts the hidden layered dimensions, has the capabilities of increasing the number of layers, and tuning the batch size affected outputs on training and validation scores. There was potential in overfitting for larger configuration, as compared to having larger models using bidirectional layers and pooling, which had really good regularization.